# 28 · Embedding the agent as a control

## Goal

Add the Copilot Studio agent to the `SupplierDetail` screen as a real chat
control, wired to the currently-selected supplier — the actual "agent
inside an app" moment this whole elective track is building toward.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
assert Path("../apps/renewal-desk-canvas/src/Screens/SupplierDetail.pa.yaml").exists(), "run 27 first"


In [ ]:
from csx.config import load_settings
settings = load_settings()
settings.require("TENANT_ID", "DATAVERSE_ENV_ID", "DELEGATED_CLIENT_ID")


## Concept

**Finding (Part 2, doc-checked 17 Aug 2026): the native "Add AI Copilot"
canvas control was deprecated 2 Feb 2026** — you can no longer add it to a
new canvas app; existing apps keep working for a limited time only. Don't
build on it. The currently-supported way to embed a Copilot Studio agent
in a canvas app is the **ChatControl PCF component** from the Copilot
Studio Samples repository: Bot Framework WebChat with a Fluent UI theme,
distributed as a Power Platform solution you import and then add like any
other control, configured with your Azure app's client ID, tenant ID, the
environment ID, and the agent identifier — the same four values
`csx/clients.py`'s delegated flow already uses, because it's the same
`CopilotStudio.Copilots.Invoke` permission underneath.

Microsoft's stated direction is Microsoft 365 Copilot in canvas apps
(public preview as of this doc-check; GA already in model-driven apps) —
worth watching, not worth building on yet. Treat this notebook's control
choice as the currently-correct one for a PREVIEW-averse build, not as
permanent; re-check before teaching this notebook again.

**Context passing, not a blank chat window:** the control's job here is to
open already knowing which supplier the user is looking at — pass the
gallery's `Selected` record as a context variable, so the first question
someone asks doesn't have to be "which supplier am I even talking about."


## Build


### Import the ChatControl PCF solution


In [ ]:
import subprocess
# The ChatControl PCF ships as a solution from the Copilot Studio Samples
# repo — import it like any other component solution.
result = subprocess.run([
    "pac", "solution", "import",
    "--path", "ChatControlPCF_solution.zip",  # downloaded from the Copilot Studio Samples repo
    "--environment", "$DATAVERSE_ENV_URL",
], capture_output=True, text=True)
print(result.returncode)


### Add the control to SupplierDetail (designer) and configure it


In the designer, on `SupplierDetail`, insert the ChatControl. Set:
- `EnvironmentId` = your Dataverse environment ID
- `AgentIdentifier` = `crd_contract-renewal-desk`
- `TenantId` / `ClientId` = the same values as `DELEGATED_CLIENT_ID`/`TENANT_ID`
- Context variable binding: `SupplierGallery.Selected.crd_SupplierName`

Save — Git Integration syncs the control's config into `SupplierDetail.pa.yaml`
(already shown, as a placeholder, in `apps/renewal-desk-canvas/src/Screens/SupplierDetail.pa.yaml`
from repo init — verify your real sync matches its shape, don't assume it does).


In [ ]:
from csx.checkpoint import checkpoint
import yaml
from pathlib import Path

def probe_chat_control():
    doc = yaml.safe_load((Path("../apps/renewal-desk-canvas/src/Screens/SupplierDetail.pa.yaml")).read_text())
    children = doc.get("Screens", {}).get("SupplierDetail", {}).get("Children", [])
    return any("AgentChat" in c or "ChatControl" in str(c) for c in children)

checkpoint(
    name="ChatControl added to SupplierDetail and synced",
    probe=probe_chat_control,
    remediation="Add the ChatControl PCF to SupplierDetail in the designer, configure its four properties, save, wait for Git Integration to sync.",
)


## Verify

Same harness, same golden set, every notebook.


There's no API-level way to drive a PCF control headlessly from here — verification is a manual smoke test, stated as such rather than faked as automated.


In [ ]:
print("Manual check: open SupplierDetail for Meridian Cables in Play mode. Confirm the chat opens already scoped to Meridian Cables — ask 'what's the notice period?' without naming the supplier and confirm it answers correctly (proves the context variable is actually wired, not just present).")


In [ ]:
from csx.clients import get_copilot_client
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter

client = get_copilot_client(settings, delegated=True)
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))
# The control calls the same published agent — the core suite is still the
# right regression floor even though the entry point changed.
suite = run_suite(client, cases=load_golden(tags=["core"]), credit_meter=meter, min_pass_rate=0.8)


## Cost


In [ ]:
meter.report_cost("28", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits, note="ChatControl PCF calls the same agent over the same API — same credit meter as any other invocation")


## Teardown


In [ ]:
print("No teardown — the embedded control persists; 29-31 add flows alongside it.")
